# 06 — Interpretability Analysis

Feature importance from GBDTs, attention patterns from Transformers, and model behavior comparison.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.registry import load_dataset
from src.data.preprocessing import preprocess_for_gbdt, preprocess_for_deep_learning
from src.models.factory import create_model
from src.utils.reproducibility import set_seed

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

set_seed(42)

In [ ]:
# Use a representative dataset
DATASET = 'adult'  # Binary classification with mixed features

X, y, info = load_dataset(DATASET)

# Quick train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset: {DATASET}")
print(f"Features: {info.num_columns + info.cat_columns}")

In [ ]:
# Train XGBoost and extract feature importance
prep = preprocess_for_gbdt(X_train, info, X_val=X_test)

xgb_model = create_model('xgboost', info.task_type, info.n_classes)
xgb_model.fit(prep.X_train, y_train)

# Get feature names from preprocessor
feature_names = info.num_columns + info.cat_columns

# Feature importance
importance = xgb_model.model.feature_importances_
imp_df = pd.DataFrame({'feature': feature_names[:len(importance)], 'importance': importance})
imp_df = imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
imp_df.plot.barh(x='feature', y='importance', ax=ax, legend=False, color='steelblue')
ax.set_title(f'XGBoost Feature Importance — {DATASET}')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_xgboost.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Compare feature importance across GBDTs
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, model_name in zip(axes, ['xgboost', 'lightgbm', 'catboost']):
    model = create_model(model_name, info.task_type, info.n_classes)
    model.fit(prep.X_train, y_train)

    importance = model.model.feature_importances_
    imp_df = pd.DataFrame({
        'feature': feature_names[:len(importance)],
        'importance': importance / importance.sum()  # Normalize
    })
    imp_df = imp_df.sort_values('importance', ascending=True).tail(15)

    imp_df.plot.barh(x='feature', y='importance', ax=ax, legend=False, color='steelblue')
    ax.set_title(f'{model_name.upper()} — Top 15 Features')
    ax.set_xlabel('Normalized Importance')

plt.suptitle(f'Feature Importance Comparison — {DATASET}', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/feature_importance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction agreement between models
from sklearn.metrics import cohen_kappa_score

models_to_compare = ['xgboost', 'lightgbm', 'catboost']
predictions = {}

for model_name in models_to_compare:
    model = create_model(model_name, info.task_type, info.n_classes)
    model.fit(prep.X_train, y_train)
    predictions[model_name] = model.predict(prep.X_test)

# Cohen's kappa between all pairs
kappa_matrix = pd.DataFrame(
    np.ones((len(models_to_compare), len(models_to_compare))),
    index=models_to_compare, columns=models_to_compare,
)
for i, m1 in enumerate(models_to_compare):
    for j, m2 in enumerate(models_to_compare):
        if i != j:
            kappa_matrix.loc[m1, m2] = cohen_kappa_score(predictions[m1], predictions[m2])

plt.figure(figsize=(8, 6))
sns.heatmap(kappa_matrix.astype(float), annot=True, fmt='.3f',
            cmap='YlGn', vmin=0.5, vmax=1.0)
plt.title(f'Prediction Agreement (Cohen\'s Kappa) — {DATASET}')
plt.tight_layout()
plt.savefig('../results/figures/prediction_agreement.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Per-sample analysis: which samples are "hard" for which family?
# Compare error patterns between best GBDT and best DL model
print("\nSamples where models disagree can reveal different inductive biases.")
print("GBDTs may handle feature interactions differently than Transformers.")
print("\nTo extend: train DL models and compare error overlap patterns.")